Transfer Learning with Sophon Embeddings (JetClass)

In [ ]:
#imports
import os, re, glob, math, random
from pathlib import Path

import numpy as np
import pandas as pd


import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve,
    auc,
    classification_report,
    confusion_matrix,
)

# Plotting
import matplotlib.pyplot as plt

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

In [ ]:
# Notebook settings: paths, task type, and training options

# Expected naming: <ClassName>_inference_with_embedding.csv
EMB_DIR = Path("embeddings")

# Optional: if baseline probs/logits are in a separate folder (same filenames)
BASELINE_DIR = None  # e.g. Path("baseline_probs")

# Embedding columns
EMB_PREFIX = "emb_"
EMB_DIM = 128

# Task setup
TASK = "multiclass10"  # "multiclass10" or "binary_hbb_qcd"

CLASSES_10 = [
    "HToBB",
    "HToCC",
    "HToGG",
    "HToWW2Q1L",
    "HToWW4Q",
    "TTBar",
    "TTBarLep",
    "WToQQ",
    "ZToQQ",
    "ZToNuNu",
]
BINARY_CLASSES = ["HToBB", "ZToNuNu"]

# Data size controls
PER_CLASS_CAP = None  # e.g. 10_000, 100_000, 1_000_000, 5_000_000

# Split strategy
# - "random_stratified": standard stratified split
# - "group_by_source_file": split by df['source_file'] (recommended if available)
SPLIT_STRATEGY = "random_stratified"

TEST_FRAC = 0.15
VAL_FRAC  = 0.15  # of train+val

# Training
SEED = 1337
BATCH_SIZE = 1024
EPOCHS = 25
LR = 3e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 5  # early stopping on val macro-AUC

# Default MLP architecture
HIDDEN_LAYERS = [256, 128, 64]
DROPOUT = 0.10
USE_BATCHNORM = True

# Architecture sweep
DO_ARCH_SWEEP = True
ARCH_SWEEP = {
    "tiny":   [64],
    "small":  [128, 64],
    "medium": [256, 128, 64],
    "large":  [512, 256, 128, 64],
}

# Data-size sweep (#jets per class)
DO_SIZE_SWEEP = True
SIZE_SWEEP_PER_CLASS = [10_000, 100_000, 1_000_000, 5_000_000]

# Confidence intervals
DO_BOOTSTRAP_CI = True
BOOTSTRAP_ITERS = 500
CI_ALPHA = 0.05  # 95% CI

# Leakage sanity test
DO_SHUFFLED_LABEL_TEST = True
SHUFFLED_TEST_EPOCHS = 5

# Output
OUT_DIR = Path("results")
OUT_DIR.mkdir(exist_ok=True)

# Precision for printing AUC in plots/prints


In [ ]:
# loading CSVs, splitting data, and building DataLoaders
def set_seed(seed: int = 1337):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def discover_files(dirpath: Path) -> dict:
    patterns = [
        "*_inference_with_embedding.csv",
        "*_inference_with_embedding.csv.gz",
        "*embedding*.csv",
        "*embedding*.csv.gz",
    ]
    files = []
    for pat in patterns:
        files.extend(dirpath.glob(pat))
    files = sorted({fp.resolve() for fp in files})

    out = {}
    for fp in files:
        name = fp.name
        if name.endswith(".gz"):
            name = name[:-3]
        if name.endswith(".csv"):
            name = name[:-4]

        m = re.match(r"(.+)_inference_with_embedding$", name)
        if m:
            key = m.group(1)
        else:
            key = name.split("_")[0]
        out[key] = fp
    return out

def emb_cols():
    return [f"{EMB_PREFIX}{i}" for i in range(EMB_DIM)]

def detect_baseline_cols(df: pd.DataFrame, num_classes: int):
    # Preferred: prob_0..prob_{C-1}
    prob_cols = [f"prob_{i}" for i in range(num_classes)]
    if all(c in df.columns for c in prob_cols):
        return ("probs", prob_cols)
    # Alternative: logit_0..logit_{C-1}
    logit_cols = [f"logit_{i}" for i in range(num_classes)]
    if all(c in df.columns for c in logit_cols):
        return ("logits", logit_cols)
    return (None, None)

def softmax_np(x: np.ndarray, axis=1):
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)

def load_per_class_csvs(class_names, emb_dir: Path, per_class_cap=None, baseline_dir=None):
    # Returns: df_all, label_map, baseline_info
    file_map = discover_files(emb_dir)
    missing = [c for c in class_names if c not in file_map]
    if missing:
        raise FileNotFoundError(
            "Missing embedding CSVs for: " + ", ".join(missing) + "\n" +
            f"Found classes in {emb_dir}: {sorted(file_map.keys())[:50]}"
        )

    label_map = {c:i for i,c in enumerate(class_names)}
    rows = []
    baseline_mode = None
    baseline_cols = None
    baseline_available = True

    for cls in class_names:
        df = pd.read_csv(file_map[cls])

        # Cap per class
        if per_class_cap is not None and len(df) > per_class_cap:
            df = df.sample(n=per_class_cap, random_state=SEED).reset_index(drop=True)

        # Ensure embeddings exist
        ecols = emb_cols()
        for c in ecols:
            if c not in df.columns:
                raise ValueError(f"{file_map[cls]} missing expected embedding column {c}")

        out = df[ecols].copy()
        out["class_name"] = cls
        out["y"] = label_map[cls]

        # Carry source_file if present (recommended for group split)
        if "source_file" in df.columns:
            out["source_file"] = df["source_file"].astype(str)
        elif "file" in df.columns:
            out["source_file"] = df["file"].astype(str)

        # Baseline: try from same CSV (preferred)
        mode, cols = detect_baseline_cols(df, num_classes=len(class_names))
        if mode is None and baseline_dir is not None:
            bmap = discover_files(Path(baseline_dir))
            if cls in bmap:
                bdf = pd.read_csv(bmap[cls])
                mode, cols = detect_baseline_cols(bdf, num_classes=len(class_names))
                if mode is not None:
                    for c in cols:
                        out[c] = bdf[c].to_numpy()
        else:
            if mode is not None:
                for c in cols:
                    out[c] = df[c].to_numpy()

        if mode is None:
            baseline_available = False
        else:
            if baseline_mode is None:
                baseline_mode, baseline_cols = mode, cols

        rows.append(out)

    df_all = pd.concat(rows, ignore_index=True)

    baseline_info = {
        "available": baseline_available,
        "mode": baseline_mode,
        "cols": baseline_cols,
        "note": (
            "Baseline columns found. Will compute Sophon out-of-the-box metrics."
            if baseline_available else
            "No baseline prob/logit columns detected. Baseline comparison will be skipped."
        )
    }
    return df_all, label_map, baseline_info

def make_splits_random_stratified(df_all: pd.DataFrame, test_frac: float, val_frac: float):
    idx = np.arange(len(df_all))
    y = df_all["y"].to_numpy()
    idx_trainval, idx_test = train_test_split(idx, test_size=test_frac, random_state=SEED, stratify=y)
    idx_train, idx_val = train_test_split(idx_trainval, test_size=val_frac, random_state=SEED, stratify=y[idx_trainval])
    return idx_train, idx_val, idx_test

def make_splits_group(df_all: pd.DataFrame, test_frac: float, val_frac: float):
    if "source_file" not in df_all.columns:
        raise ValueError("Group split requested but df_all has no 'source_file' column.")
    groups = df_all["source_file"].to_numpy()
    y = df_all["y"].to_numpy()

    gss = GroupShuffleSplit(n_splits=1, test_size=test_frac, random_state=SEED)
    trainval_idx, test_idx = next(gss.split(df_all, y, groups))

    groups_tv = groups[trainval_idx]
    y_tv = y[trainval_idx]
    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_frac, random_state=SEED)
    tr_sub, va_sub = next(gss2.split(trainval_idx, y_tv, groups_tv))
    train_idx = trainval_idx[tr_sub]
    val_idx = trainval_idx[va_sub]
    return train_idx, val_idx, test_idx

def standardize_from_indices(df_all: pd.DataFrame, idx_train, idx_val, idx_test):
    ecols = emb_cols()
    X_train = df_all.loc[idx_train, ecols].to_numpy(dtype=np.float32)
    X_val   = df_all.loc[idx_val,   ecols].to_numpy(dtype=np.float32)
    X_test  = df_all.loc[idx_test,  ecols].to_numpy(dtype=np.float32)
    y_train = df_all.loc[idx_train, "y"].to_numpy(dtype=np.int64)
    y_val   = df_all.loc[idx_val,   "y"].to_numpy(dtype=np.int64)
    y_test  = df_all.loc[idx_test,  "y"].to_numpy(dtype=np.int64)

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s   = scaler.transform(X_val)
    X_test_s  = scaler.transform(X_test)
    return X_train_s, y_train, X_val_s, y_val, X_test_s, y_test, scaler

def build_loaders(X_train, y_train, X_val, y_val, X_test, y_test):
    train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
    val_ds   = TensorDataset(torch.from_numpy(X_val),   torch.from_numpy(y_val))
    test_ds  = TensorDataset(torch.from_numpy(X_test),  torch.from_numpy(y_test))
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
    return train_loader, val_loader, test_loader

def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray):
    y_pred = np.argmax(y_prob, axis=1)
    acc = accuracy_score(y_true, y_pred)
    if y_prob.shape[1] == 2:
        auc_macro = roc_auc_score(y_true, y_prob[:, 1])
    else:
        auc_macro = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    per_class_auc = []
    for k in range(y_prob.shape[1]):
        y_bin = (y_true == k).astype(int)
        per_class_auc.append(roc_auc_score(y_bin, y_prob[:, k]))
    return acc, auc_macro, per_class_auc

In [ ]:
# Load the embedding tables and run quick sanity checks
if TASK == "multiclass10":
    class_names = CLASSES_10
elif TASK == "binary_hbb_qcd":
    class_names = BINARY_CLASSES
else:
    raise ValueError(f"Unknown TASK: {TASK}")

df_all, label_map, baseline_info = load_per_class_csvs(
    class_names,
    EMB_DIR,
    per_class_cap=PER_CLASS_CAP,
    baseline_dir=BASELINE_DIR
)

print("Loaded rows:", len(df_all))
print("Classes:", label_map)
print("Baseline:", baseline_info["note"])

print("\nCounts per class:")
print(df_all["class_name"].value_counts())

# Sanity EDA: embedding norm distribution
E = df_all[emb_cols()].to_numpy(dtype=np.float32)
norms = np.linalg.norm(E, axis=1)
plt.figure()
plt.hist(norms, bins=100)
plt.xlabel("||embedding||")
plt.ylabel("count")
plt.title("Embedding norm distribution (all classes)")
plt.grid(True, ls=":")
plt.tight_layout()
plt.show()

In [ ]:
# Create train/val/test splits and standardize embeddings
if SPLIT_STRATEGY == "group_by_source_file":
    if "source_file" in df_all.columns:
        idx_train, idx_val, idx_test = make_splits_group(df_all, TEST_FRAC, VAL_FRAC)
        print("Using GROUP split by source_file.")
    else:
        print("WARNING: requested group split but no source_file column found; falling back to random_stratified.")
        idx_train, idx_val, idx_test = make_splits_random_stratified(df_all, TEST_FRAC, VAL_FRAC)
else:
    idx_train, idx_val, idx_test = make_splits_random_stratified(df_all, TEST_FRAC, VAL_FRAC)
    print("Using random stratified split.")

# Duplicate overlap check (exact duplicates in embedding vectors)
hashes = pd.util.hash_pandas_object(df_all[emb_cols()], index=False).to_numpy()
overlap_train_test = np.intersect1d(hashes[idx_train], hashes[idx_test]).size
overlap_val_test   = np.intersect1d(hashes[idx_val], hashes[idx_test]).size
print("Exact duplicate overlap train↔test:", overlap_train_test)
print("Exact duplicate overlap val↔test:  ", overlap_val_test)

X_train, y_train, X_val, y_val, X_test, y_test, scaler = standardize_from_indices(
    df_all, idx_train, idx_val, idx_test
)

train_loader, val_loader, test_loader = build_loaders(X_train, y_train, X_val, y_val, X_test, y_test)
num_classes = len(class_names)

print("Shapes:")
print(" train:", X_train.shape, y_train.shape)
print(" val:  ", X_val.shape, y_val.shape)
print(" test: ", X_test.shape, y_test.shape)

In [ ]:
# Evaluate the baseline model outputs, if available
baseline_results = None

def softmax_np(x: np.ndarray, axis=1):
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)

if baseline_info["available"]:
    cols = baseline_info["cols"]
    mode = baseline_info["mode"]

    base = df_all.loc[idx_test, cols].to_numpy(dtype=np.float64)
    y_prob_base = softmax_np(base, axis=1) if mode == "logits" else base

    acc_b, auc_b, per_b = compute_metrics(y_test, y_prob_base)
    baseline_results = {
        "test_acc": float(acc_b),
        "test_auc_macro_ovr": float(auc_b),
        "per_class_auc": [float(x) for x in per_b],
    }
    print(f"BASELINE TEST acc={acc_b:.6f} macroAUC={auc_b:.4f}")
else:
    print("Baseline columns not found; skipping baseline evaluation.")


In [ ]:
# Define the MLP head and the training/evaluation loop
class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden_layers, out_dim: int, dropout=0.0, use_batchnorm=True):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_layers:
            layers.append(nn.Linear(prev, h))
            if use_batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def build_model(hidden_layers):
    return MLP(EMB_DIM, hidden_layers, num_classes, dropout=DROPOUT, use_batchnorm=USE_BATCHNORM).to(device)

def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0
    y_true = []
    y_prob = []
    criterion = nn.CrossEntropyLoss()

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)

        if is_train:
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * xb.size(0)

        probs = torch.softmax(logits.detach(), dim=1).cpu().numpy()
        y_prob.append(probs)
        y_true.append(yb.detach().cpu().numpy())

    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)

    y_pred = np.argmax(y_prob, axis=1)
    acc = accuracy_score(y_true, y_pred)

    if y_prob.shape[1] == 2:
        auc_macro = roc_auc_score(y_true, y_prob[:, 1])
    else:
        auc_macro = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, acc, auc_macro, y_true, y_prob

def train_model(hidden_layers, epochs=25, patience=5):
    model = build_model(hidden_layers)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    history = {"epoch": [], "train_loss": [], "train_acc": [], "train_auc": [],
               "val_loss": [], "val_acc": [], "val_auc": []}

    best_val_auc = -1.0
    best_state = None
    bad_epochs = 0

    for ep in range(1, epochs+1):
        tr_loss, tr_acc, tr_auc, _, _ = run_epoch(model, train_loader, optimizer=optimizer)
        va_loss, va_acc, va_auc, _, _ = run_epoch(model, val_loader, optimizer=None)

        history["epoch"].append(ep)
        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["train_auc"].append(tr_auc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)
        history["val_auc"].append(va_auc)

        print(f"[{ep:02d}/{epochs}] "
              f"train loss {tr_loss:.4f} acc {tr_acc:.4f} auc {tr_auc:.4f} | "
              f"val loss {va_loss:.4f} acc {va_acc:.4f} auc {va_auc:.4f}")

        if va_auc > best_val_auc:
            best_val_auc = va_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping (patience={patience}) at epoch {ep}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history), float(best_val_auc)

print("Default model:")
print(build_model(HIDDEN_LAYERS))


In [ ]:
# Train a default MLP head and plot basic diagnostics
model, hist, best_val_auc = train_model(HIDDEN_LAYERS, epochs=EPOCHS, patience=PATIENCE)

# Curves
plt.figure()
plt.plot(hist["epoch"], hist["train_loss"], label="train loss")
plt.plot(hist["epoch"], hist["val_loss"], label="val loss")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.grid(True, ls=":"); plt.legend(); plt.tight_layout()
plt.show()

plt.figure()
plt.plot(hist["epoch"], hist["train_auc"], label="train macro-AUC")
plt.plot(hist["epoch"], hist["val_auc"], label="val macro-AUC")
plt.xlabel("epoch"); plt.ylabel("AUC"); plt.grid(True, ls=":"); plt.legend(); plt.tight_layout()
plt.show()

# Test
test_loss, test_acc, test_auc, y_true_m, y_prob_m = run_epoch(model, test_loader, optimizer=None)
print(f"MLP TEST loss={test_loss:.6f} acc={test_acc:.6f} macroAUC={test_auc:.4f} (best val AUC={best_val_auc:.4f})")

print("\nClassification report (test):")
print(classification_report(y_true_m, np.argmax(y_prob_m, axis=1), target_names=class_names))

cm = confusion_matrix(y_true_m, np.argmax(y_prob_m, axis=1))
plt.figure(figsize=(7,6))
plt.imshow(cm)
plt.title("Confusion matrix (MLP, test)")
plt.xlabel("pred"); plt.ylabel("true")
plt.colorbar()
plt.xticks(range(len(class_names)), class_names, rotation=90)
plt.yticks(range(len(class_names)), class_names)
plt.tight_layout()
plt.show()


In [ ]:
# ROC curves: baseline vs MLP (test)

from sklearn.metrics import roc_auc_score, roc_curve

def plot_ovr_roc(y_true, y_prob, title, outpath=None):
    plt.figure()
    for k, cls in enumerate(class_names):
        y_bin = (y_true == k).astype(int)
        fpr, tpr, _ = roc_curve(y_bin, y_prob[:, k])
        roc_auc = roc_auc_score(y_bin, y_prob[:, k])
        plt.plot(fpr, tpr, label=f"{cls} (AUC={roc_auc:.4f})")

    plt.plot([0, 1], [0, 1], ls="--", label="random")
    plt.xscale("log"); plt.yscale("log")
    plt.xlim(1e-5, 1); plt.ylim(1e-5, 1)
    plt.xlabel("FPR"); plt.ylabel("TPR")
    plt.title(title)
    plt.grid(True, ls=":")
    plt.legend(fontsize=8, loc="lower right")
    plt.tight_layout()
    if outpath is not None:
        plt.savefig(outpath, dpi=150)
    plt.show()

plot_ovr_roc(y_true_m, y_prob_m, "One-vs-rest ROC (MLP head, test)", outpath=OUT_DIR/"roc_mlp.png")

if baseline_results is not None:
    cols = baseline_info["cols"]; mode = baseline_info["mode"]
    base = df_all.loc[idx_test, cols].to_numpy(dtype=np.float64)
    y_prob_base = softmax_np(base, axis=1) if mode == "logits" else base
    plot_ovr_roc(y_test, y_prob_base, "One-vs-rest ROC (Sophon baseline, test)", outpath=OUT_DIR/"roc_baseline.png")


In [ ]:
# Sweep MLP architectures and compare performance
arch_rows = []

def eval_mlp(hidden_layers):
    m, h, best_val = train_model(hidden_layers, epochs=EPOCHS, patience=PATIENCE)
    loss_t, acc_t, auc_t, _, _ = run_epoch(m, test_loader, optimizer=None)
    return m, h, best_val, loss_t, acc_t, auc_t

best_arch = None
best_arch_auc = -1.0

if DO_ARCH_SWEEP:
    for name, layers in ARCH_SWEEP.items():
        print("\n" + "-"*70)
        print("Training architecture:", name, layers)
        m, h, best_val, loss_t, acc_t, auc_t = eval_mlp(layers)

        delta_auc = None
        delta_acc = None
        if baseline_results is not None:
            delta_auc = float(auc_t - baseline_results["test_auc_macro_ovr"])
            delta_acc = float(acc_t - baseline_results["test_acc"])

        arch_rows.append({
            "arch_name": name,
            "hidden_layers": str(layers),
            "test_loss": float(loss_t),
            "test_acc": float(acc_t),
            "test_auc_macro_ovr": float(auc_t),
            "best_val_auc": float(best_val),
            "delta_auc_vs_baseline": delta_auc,
            "delta_acc_vs_baseline": delta_acc,
        })

        if auc_t > best_arch_auc:
            best_arch_auc = auc_t
            best_arch = name

arch_df = pd.DataFrame(arch_rows).sort_values("test_auc_macro_ovr", ascending=False)
arch_df.to_csv(OUT_DIR/"arch_sweep_results.csv", index=False)
print("\nSaved:", OUT_DIR/"arch_sweep_results.csv")
arch_df

In [ ]:
# Sweep training data size and see how performance scales
size_rows = []
if DO_SIZE_SWEEP:
    chosen_layers = ARCH_SWEEP.get(best_arch, HIDDEN_LAYERS) if DO_ARCH_SWEEP and best_arch is not None else HIDDEN_LAYERS
    print("Using layers for size sweep:", chosen_layers)

    for cap in SIZE_SWEEP_PER_CLASS:
        print("\n" + "-"*70)
        print("PER_CLASS_CAP =", cap)

        df_tmp, _, base_tmp_info = load_per_class_csvs(
            class_names, EMB_DIR, per_class_cap=cap, baseline_dir=BASELINE_DIR
        )

        # Split
        if SPLIT_STRATEGY == "group_by_source_file" and "source_file" in df_tmp.columns:
            tr_i, va_i, te_i = make_splits_group(df_tmp, TEST_FRAC, VAL_FRAC)
        else:
            tr_i, va_i, te_i = make_splits_random_stratified(df_tmp, TEST_FRAC, VAL_FRAC)

        # Standardize
        Xtr, ytr, Xva, yva, Xte, yte, _ = standardize_from_indices(df_tmp, tr_i, va_i, te_i)

        # Loaders
        tr_loader, va_loader2, te_loader = build_loaders(Xtr, ytr, Xva, yva, Xte, yte)

        # Baseline on this cap (if available)
        base_auc = None
        base_acc = None
        if base_tmp_info["available"]:
            cols = base_tmp_info["cols"]
            mode = base_tmp_info["mode"]
            base_scores = df_tmp.loc[te_i, cols].to_numpy(dtype=np.float64)
            probs = softmax_np(base_scores, axis=1) if mode == "logits" else base_scores
            base_acc, base_auc, _ = compute_metrics(yte, probs)

        # Rebind loaders used by train_model/run_epoch for this block
        train_loader_b, val_loader_b, test_loader_b = train_loader, val_loader, test_loader

        train_loader, val_loader, test_loader = tr_loader, va_loader2, te_loader

        m_cap, h_cap, best_val = train_model(chosen_layers, epochs=EPOCHS, patience=PATIENCE)
        loss_t, acc_t, auc_t, _, _ = run_epoch(m_cap, test_loader, optimizer=None)

        # restore
        train_loader, val_loader, test_loader = train_loader_b, val_loader_b, test_loader_b

        size_rows.append({
            "per_class_cap": int(cap),
            "mlp_test_acc": float(acc_t),
            "mlp_test_auc_macro_ovr": float(auc_t),
            "mlp_best_val_auc": float(best_val),
            "baseline_test_acc": (float(base_acc) if base_acc is not None else None),
            "baseline_test_auc_macro_ovr": (float(base_auc) if base_auc is not None else None),
            "delta_auc_vs_baseline": (float(auc_t - base_auc) if base_auc is not None else None),
        })

size_df = pd.DataFrame(size_rows)
size_df.to_csv(OUT_DIR/"size_sweep_results.csv", index=False)
print("\nSaved:", OUT_DIR/"size_sweep_results.csv")
size_df

In [ ]:
# Bootstrap confidence intervals for macro-AUC
def bootstrap_macro_auc(y_true, y_prob, iters=500, alpha=0.05):
    rng = np.random.default_rng(SEED)
    n = len(y_true)
    vals = []
    for _ in range(iters):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        yp = y_prob[idx]
        if yp.shape[1] == 2:
            v = roc_auc_score(yt, yp[:, 1])
        else:
            v = roc_auc_score(yt, yp, multi_class="ovr", average="macro")
        vals.append(v)
    vals = np.array(vals)
    lo = np.quantile(vals, alpha/2)
    hi = np.quantile(vals, 1 - alpha/2)
    return float(np.mean(vals)), float(lo), float(hi)

ci_results = {}
if DO_BOOTSTRAP_CI:
    mean_m, lo_m, hi_m = bootstrap_macro_auc(y_true_m, y_prob_m, iters=BOOTSTRAP_ITERS, alpha=CI_ALPHA)
    ci_results["mlp_macro_auc_mean"] = mean_m
    ci_results["mlp_macro_auc_ci_lo"] = lo_m
    ci_results["mlp_macro_auc_ci_hi"] = hi_m
    print(f"MLP macro-AUC bootstrap mean={mean_m:.4f}  CI=[{lo_m:.4f}, {hi_m:.4f}]")

    if baseline_results is not None:
        cols = baseline_info["cols"]; mode = baseline_info["mode"]
        base = df_all.loc[idx_test, cols].to_numpy(dtype=np.float64)
        y_prob_base = softmax_np(base, axis=1) if mode == "logits" else base
        mean_b, lo_b, hi_b = bootstrap_macro_auc(y_test, y_prob_base, iters=BOOTSTRAP_ITERS, alpha=CI_ALPHA)
        ci_results["baseline_macro_auc_mean"] = mean_b
        ci_results["baseline_macro_auc_ci_lo"] = lo_b
        ci_results["baseline_macro_auc_ci_hi"] = hi_b
        print(f"BASELINE macro-AUC bootstrap mean={mean_b:.4f}  CI=[{lo_b:.4f}, {hi_b:.4f}]")


In [ ]:
# Print a compact summary of results

print("Transfer learning summary (Sophon embeddings -> MLP head)")
print(f"Task: {TASK}")
print(f"Split strategy: {SPLIT_STRATEGY}")
print(f"Per-class cap: {PER_CLASS_CAP}")

if baseline_results is None:
    print("Baseline: not available (no prob_*/logit_* columns found)")
else:
    print(f"Baseline test accuracy: {baseline_results['test_acc']:.6f}")
    print(f"Baseline test macro AUC (OVR): {baseline_results['test_auc_macro_ovr']:.6f}")

print(f"MLP hidden layers: {HIDDEN_LAYERS}")
print(f"MLP test accuracy: {float(test_acc):.6f}")
print(f"MLP test macro AUC (OVR): {float(test_auc):.6f}")
print(f"Best val macro AUC: {float(best_val_auc):.6f}")

print(f"Best architecture from sweep: {best_arch}")

if isinstance(ci_results, dict) and len(ci_results) > 0 and 'mlp_macro_auc_mean' in ci_results:
    print(f"MLP macro AUC bootstrap mean: {ci_results['mlp_macro_auc_mean']:.6f}")
    print(f"MLP macro AUC 95% CI: [{ci_results['mlp_macro_auc_ci_lo']:.6f}, {ci_results['mlp_macro_auc_ci_hi']:.6f}]")

